MovieLens Dataset Exploration

In [1]:
import pandas as pd
from pathlib import Path

Locate the dataset

In [2]:
DATA_PATH = Path("../../data/")

movies_path = DATA_PATH / "movies.csv"
ratings_path = DATA_PATH / "ratings.csv"

print("Movies file exists:", movies_path.exists())
print("Ratings file exists:", ratings_path.exists())

Movies file exists: True
Ratings file exists: True


Load the datasets

In [3]:
# Load movie information
movies_df = pd.read_csv(movies_path)

# Load user ratings
ratings_df = pd.read_csv(ratings_path)

print("Movies Dataset:")
display(movies_df.head())

print("Ratings Dataset:")
display(ratings_df.head())

Movies Dataset:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Ratings Dataset:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


Dataset dimensions

In [4]:
print("Movies shape:", movies_df.shape)
print("Ratings shape:", ratings_df.shape)

Movies shape: (9742, 3)
Ratings shape: (100836, 4)


Column information

In [5]:
print("Movies information:")
movies_df.info()

print("\nRatings information:")
ratings_df.info()

Movies information:
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 228.5 KB

Ratings information:
<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


Statistical summary

In [6]:
print("Movies summary:")
display(movies_df.describe(include="all"))

print("Ratings summary:")
display(ratings_df.describe())

Movies summary:


,movieId,title,genres
count,9742.000000,9742,9742
unique,NaN,9737,951
top,NaN,Emma (1996),Drama
freq,NaN,2,1053
mean,42200.353623,NaN,NaN
std,52160.494854,NaN,NaN
min,1.000000,NaN,NaN
25%,3248.250000,NaN,NaN
50%,7300.000000,NaN,NaN
75%,76232.000000,NaN,NaN


Ratings summary:


,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


Count unique users and movies

In [7]:
total_users = ratings_df["userId"].nunique()
total_movies = movies_df["movieId"].nunique()
total_ratings = len(ratings_df)

print("Total users:", total_users)
print("Total movies:", total_movies)
print("Total ratings:", total_ratings)

Total users: 610
Total movies: 9742
Total ratings: 100836


Understand rating distribution

In [8]:
print(ratings_df["rating"].value_counts().sort_index())

rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64


Calculate movie statistics

In [9]:
movie_stats=ratings_df.groupby("movieId").agg(average_rating=("rating","mean"),rating_count=("rating","count")).reset_index()
movie_stats.head()

,movieId,average_rating,rating_count
0,1,3.920930,215
1,2,3.431818,110
2,3,3.259615,52
3,4,2.357143,7
4,5,3.071429,49


- C = Average rating across all movies
- m = Minimum rating count (90th percentile)

In [10]:
C=ratings_df["rating"].mean()
m=movie_stats["rating_count"].quantile(0.90)

print("Overall average rating:", C)
print("Minimum rating count:", m)

Overall average rating: 3.501556983616962
Minimum rating count: 27.0


Calculate weighted rating

In [11]:
# Calculate weighted rating for every movie
movie_stats["weighted_rating"] = (
    (movie_stats["rating_count"] / (movie_stats["rating_count"] + m))
    * movie_stats["average_rating"]
    +
    (m / (movie_stats["rating_count"] + m))
    * C
)

movie_stats.head()

,movieId,average_rating,rating_count,weighted_rating
0,1,3.920930,215,3.874141
1,2,3.431818,110,3.445562
2,3,3.259615,52,3.342304
3,4,2.357143,7,3.265942
4,5,3.071429,49,3.224237


Merge movie information

In [12]:
popular_movies=movies_df.merge(movie_stats,on="movieId",how="inner")
popular_movies.head()

,movieId,title,genres,average_rating,rating_count,weighted_rating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215,3.874141
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.431818,110,3.445562
2,3,Grumpier Old Men (1995),Comedy|Romance,3.259615,52,3.342304
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143,7,3.265942
4,5,Father of the Bride Part II (1995),Comedy,3.071429,49,3.224237


Generate recommendations

In [13]:
# Recommend movies with sufficient ratings

popular_movies = popular_movies[
    popular_movies["rating_count"] >= m
]

# Sort by weighted rating
popular_movies=popular_movies.sort_values(by="weighted_rating",ascending=False)

# Display Top 10 recommendations

top_10_movies = popular_movies[
    ["title", "genres", "average_rating", "rating_count", "weighted_rating"]
].head(10)

top_10_movies

,title,genres,average_rating,rating_count,weighted_rating
277,"Shawshank Redemption, The (1994)",Crime|Drama,4.429022,317,4.356227
659,"Godfather, The (1972)",Crime|Drama,4.289062,192,4.191973
2224,Fight Club (1999),Action|Crime|Drama|Thriller,4.272936,218,4.187927
224,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.231076,251,4.160223
46,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.237745,204,4.151697
461,Schindler's List (1993),Drama|War,4.225000,220,4.145919
257,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.197068,307,4.140844
897,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,4.215640,211,4.134630
1938,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.192446,278,4.131285
921,"Godfather: Part II, The (1974)",Crime|Drama,4.259690,129,4.128475


Tags Dataset

In [14]:
tags = pd.read_csv(DATA_PATH / "tags.csv")

print("TAGS DATASET")
print(tags.head())
print(tags.info())
print("Shape:", tags.shape)
print("Columns:", tags.columns.tolist())

TAGS DATASET
   userId  movieId              tag   timestamp
0       2    60756            funny  1445714994
1       2    60756  Highly quotable  1445714996
2       2    60756     will ferrell  1445714992
3       2    89774     Boxing story  1445715207
4       2    89774              MMA  1445715200
<class 'pandas.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   userId     3683 non-null   int64
 1   movieId    3683 non-null   int64
 2   tag        3683 non-null   str  
 3   timestamp  3683 non-null   int64
dtypes: int64(3), str(1)
memory usage: 115.2 KB
None
Shape: (3683, 4)
Columns: ['userId', 'movieId', 'tag', 'timestamp']


User-Based Collaborative Filtering

In [16]:
from sklearn.metrics.pairwise import cosine_similarity
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine=create_engine(os.getenv("DATABASE_URL"))
ratings_df=pd.read_sql(
    """
    SELECT dataset_user_id, movie_id, rating
    FROM ratings
    WHERE dataset_user_id IS NOT NULL
    """,
    engine
)

ratings_df.head()

,dataset_user_id,movie_id,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


Interaction matrix

In [ ]:
user_item_matrix=ratings_df.pivot_table(
    index="dataset_user_id", #row
    columns="movie_id", #col
    values="rating"
)
user_item_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
dataset_user_id,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
print("Matrix shape:", user_item_matrix.shape)
print("Total users:", user_item_matrix.shape[0])
print("Total movies:", user_item_matrix.shape[1])
print("Missing values:", user_item_matrix.isnull().sum().sum())

Matrix shape: (610, 9724)
Total users: 610
Total movies: 9724
Missing values: 5830804


In [19]:
#Replacing missing ratings with 0 for cosine similarity
user_item_filled=user_item_matrix.fillna(0)
user_item_filled.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
dataset_user_id,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Calculate user similarity

In [20]:
user_similarity=cosine_similarity(user_item_filled)

user_similarity_df=pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)
user_similarity_df.head()

dataset_user_id,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
dataset_user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


Finding similar users

In [21]:
user_id=1

similar_users=user_similarity_df[user_id].sort_values(ascending=False)

#Exclude user themselves
similar_users=similar_users.drop(user_id)
similar_users.head(5)

dataset_user_id
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
Name: 1, dtype: float64

Generate recommendations

In [36]:
def recommend_movies(user_id,top_n=10,similar_users_count=10):
    #get similar users
    similar_users=user_similarity_df[user_id].sort_values(ascending=False).drop(user_id)
    similar_users=similar_users.head(similar_users_count)
    
    #get movies the target user hasn't rated
    user_ratings=user_item_matrix.loc[user_id]
    
    unrated_movies=user_ratings[user_ratings.isna()].index
    
    #collect recommendations from similar users
    recommendations={}
    
    for similar_user_id,similarity_score in similar_users.items():
        # Get similar user's ratings
        ratings=user_item_matrix.loc[similar_user_id]
        
        for movie_id in unrated_movies:
            rating=ratings[movie_id]
            
            if pd.notna(rating) and rating>=4:
                if movie_id not in recommendations:
                    recommendations[movie_id]=0
                recommendations[movie_id]+=similarity_score* rating
                
    #sort recommendations
    recommendations=sorted(recommendations.items(),key=lambda x:x[1], reverse=True)
    
    return recommendations[:top_n]

In [37]:
recommendations = recommend_movies(user_id=1, top_n=10)

recommendations

[(589, np.float64(13.146917710802267)),
 (858, np.float64(13.134549527466067)),
 (1200, np.float64(12.936063266381074)),
 (2762, np.float64(12.432916270630933)),
 (1221, np.float64(12.104994580853713)),
 (541, np.float64(11.55601355118029)),
 (1036, np.float64(10.88940948820309)),
 (1653, np.float64(10.561736783572758)),
 (1610, np.float64(10.342091134483228)),
 (32, np.float64(10.332388535018081))]

Display movie titles

In [38]:
recommended_movie_ids = [
    movie_id for movie_id, score in recommendations
]

recommended_movies=pd.read_sql(
    """
    SELECT movie_id,title,genres
    FROM movies
    """,
    engine
)

recommended_movies=recommended_movies[recommended_movies["movie_id"].isin(recommended_movie_ids)]
recommended_movies

,movie_id,title,genres
26,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
467,541,Blade Runner (1982),Action|Sci-Fi|Thriller
498,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi
649,858,"Godfather, The (1972)",Crime|Drama
780,1036,Die Hard (1988),Action|Crime|Thriller
888,1200,Aliens (1986),Action|Adventure|Horror|Sci-Fi
908,1221,"Godfather: Part II, The (1974)",Crime|Drama
1196,1610,"Hunt for Red October, The (1990)",Action|Adventure|Thriller
1228,1653,Gattaca (1997),Drama|Sci-Fi|Thriller
2051,2762,"Sixth Sense, The (1999)",Drama|Horror|Mystery
